# simple: настройка последнего слоя ResNet18

## 1. Данные и то же разбиение

In [ ]:
%pip install -q iterative-stratification

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet18
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from tqdm.auto import tqdm

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'data' / 'train.csv').exists())
TRAIN_DIR = ROOT / 'data' / 'train'
MODEL_DIR = ROOT / 'labels' / 'simple' / 'models'
CHECKPOINT_PATH = MODEL_DIR / 'best_simple_resnet18.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train = pd.read_csv(ROOT / 'data' / 'train.csv')
target_columns = train.columns.drop('item_id').tolist()
splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_index, _ = next(splitter.split(train, train[target_columns]))
train_val_df = train.iloc[train_val_index].reset_index(drop=True)
splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1765, random_state=43)
train_index, val_index = next(splitter.split(train_val_df, train_val_df[target_columns]))
train_df = train_val_df.iloc[train_index].reset_index(drop=True)
val_df = train_val_df.iloc[val_index].reset_index(drop=True)

print('Train:', len(train_df), 'Validation:', len(val_df), 'Устройство:', device)

## 2. Признаки из сохранённой ResNet18

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 336)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class ImageDataset(Dataset):
    def __init__(self, data):
        self.paths = [TRAIN_DIR / f'{item_id}.png' for item_id in data['item_id']]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return transform(image.convert('RGB'))

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
assert checkpoint['target'] == 'simple'
assert checkpoint['target_columns'] == target_columns
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
model.load_state_dict(checkpoint['model_state_dict'])
model.fc = nn.Identity()
model = model.to(device).eval()

def extract_features(data):
    loader = DataLoader(ImageDataset(data), batch_size=32, shuffle=False,
                        num_workers=0, pin_memory=device.type == 'cuda')
    features = []
    with torch.inference_mode():
        for images in tqdm(loader):
            features.append(model(images.to(device)).cpu().numpy())
    return np.concatenate(features)

X_train = extract_features(train_df)
X_val = extract_features(val_df)
y_train = train_df['simple'].to_numpy()
y_val = val_df['simple'].to_numpy()
print('Размер признаков:', X_train.shape, X_val.shape)

## 3. Подбор регуляризации последнего слоя

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.01)
results = []
heads = {}

for c in [0.01, 0.1, 1.0]:
    head = LogisticRegression(C=c, class_weight='balanced',
                              solver='liblinear', max_iter=1000, random_state=42)
    head.fit(X_train, y_train)
    probabilities = head.predict_proba(X_val)[:, 1]
    scores = [f1_score(y_val, probabilities >= t) for t in thresholds]
    threshold = float(thresholds[np.argmax(scores)])
    prediction = probabilities >= threshold
    results.append({
        'C': c, 'AP': average_precision_score(y_val, probabilities),
        'threshold': round(threshold, 2), 'F1': f1_score(y_val, prediction),
        'precision': precision_score(y_val, prediction),
        'recall': recall_score(y_val, prediction),
        'FP': int(((y_val == 0) & prediction).sum()),
        'FN': int(((y_val == 1) & ~prediction).sum())
    })
    heads[c] = head

results = pd.DataFrame(results).sort_values('F1', ascending=False)
display(results.round(4))
print('F1 текущей ResNet18:', 0.7014)
print('F1 прежней связки:', 0.7090)

## 4. Сохранение лучшего варианта

In [ ]:
best = results.iloc[0]
if best['F1'] > 0.7014492753623188:
    joblib.dump(heads[best['C']], MODEL_DIR / 'simple_resnet_head.joblib')
    with open(MODEL_DIR / 'simple_resnet_head.json', 'w', encoding='utf-8') as file:
        json.dump({'C': float(best['C']), 'threshold': float(best['threshold']),
                   'validation_f1': float(best['F1']), 'seed': 42}, file, indent=2)
    print('Новый вариант сохранён. F1:', round(best['F1'], 4))
else:
    print('Прироста нет. Оставляем прежнюю модель.')